In [7]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
!pip install scikit-learn joblib pyodbc xgboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import sklearn
print("sklearn_version:",sklearn.__version__)

sklearn_version: 1.8.0


In [13]:
# Loading Feature Matrix
import pandas as pd
import numpy as np

df=pd.read_csv("fraud_feature_matrix_v1.csv")

y=df["fraud_label"].astype(int)
X=df.drop(columns=["fraud_label"])

print("X_shape:",X.shape)
print("fraud_rate_pct:",round(y.mean()*100,2))

X_shape: (5000, 25)
fraud_rate_pct: 23.42


In [16]:
# Finding which columns are not numeric
import pandas as pd
import numpy as np

df=pd.read_csv("fraud_feature_matrix_v1.csv")

y=df["fraud_label"].astype(int)
X=df.drop(columns=["fraud_label"])

non_numeric=X.select_dtypes(exclude=["number"]).columns.tolist()
print("non_numeric_cols:",non_numeric)
print(X[non_numeric].head(3).to_string(index=False))

non_numeric_cols: ['transaction_timestamp', 'currency', 'merchant_category', 'payment_method', 'channel', 'device_type', 'customer_region', 'transaction_datetime']
transaction_timestamp currency merchant_category payment_method    channel device_type customer_region transaction_datetime
  2023-06-08 16:57:00      INR            Travel         Wallet Mobile App         Web             UAE  2023-06-08 16:57:00
  2023-04-23 11:53:00      INR           Grocery     Debit Card        POS      Tablet         Germany  2023-04-23 11:53:00
  2023-07-25 20:22:00      INR            Luxury            UPI     Online      Tablet             UAE  2023-07-25 20:22:00


In [22]:
# Converting datetime columns into numeric features
import pandas as pd
import numpy as np

df=pd.read_csv("fraud_feature_matrix_v1.csv")

y=df["fraud_label"].astype(int)
X=df.drop(columns=["fraud_label"])

if "transaction_datetime" in X.columns:
    X["transaction_datetime"]=pd.to_datetime(X["transaction_datetime"],errors="coerce")
    X["txn_hour"]=X["transaction_datetime"].dt.hour
    X["txn_dayofweek"]=X["transaction_datetime"].dt.dayofweek
    X["is_weekend"]=(X["txn_dayofweek"]>=5).astype(int)
    X=X.drop(columns=["transaction_datetime"])

if "transaction_date" in X.columns:
    X["transaction_date"]=pd.to_datetime(X["transaction_date"],errors="coerce")
    X["txn_month"]=X["transaction_date"].dt.month
    X["txn_day"]=X["transaction_date"].dt.day
    X=X.drop(columns=["transaction_date"])

non_numeric=X.select_dtypes(exclude=["number"]).columns.tolist()
print("remaining_non_numeric_cols:",non_numeric)

X=X.apply(pd.to_numeric,errors="coerce")
X=X.fillna(0)

print("X_shape:",X.shape)
print("nulls_total:",int(X.isna().sum().sum()))

remaining_non_numeric_cols: ['transaction_timestamp', 'currency', 'merchant_category', 'payment_method', 'channel', 'device_type', 'customer_region']
X_shape: (5000, 25)
nulls_total: 0


In [20]:
# Train/Test Split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.3,random_state=42,stratify=y
)

print("train:",X_train.shape,"test:",X_test.shape)

train: (3500, 25) test: (1500, 25)


In [24]:
# Logistic Regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,average_precision_score

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

lr_pipe=Pipeline([
    ("scaler",StandardScaler(with_mean=False)),
    ("lr",LogisticRegression(max_iter=5000,class_weight="balanced",solver="lbfgs"))
])

lr_pipe.fit(X_train,y_train)

lr_prob=lr_pipe.predict_proba(X_test)[:,1]

print("LR_SCALED_ROC_AUC:",round(roc_auc_score(y_test,lr_prob),4))
print("LR_SCALED_PR_AUC:",round(average_precision_score(y_test,lr_prob),4))

LR_SCALED_ROC_AUC: 0.7256
LR_SCALED_PR_AUC: 0.5685


In [26]:
# Strong Model: Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score,average_precision_score

rf=RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced",
    min_samples_leaf=5
)

rf.fit(X_train,y_train)

rf_prob=rf.predict_proba(X_test)[:,1]

print("RF_ROC_AUC:",round(roc_auc_score(y_test,rf_prob),4))
print("RF_PR_AUC:",round(average_precision_score(y_test,rf_prob),4))

RF_ROC_AUC: 0.7002
RF_PR_AUC: 0.5588


In [27]:
# Threshold Tuning
import numpy as np
from sklearn.metrics import f1_score

thresholds=np.round(np.arange(0.05,0.96,0.01),2)

best_t=0.5
best_f1=-1

for t in thresholds:
    pred=(rf_prob>=t).astype(int)
    f1=f1_score(y_test,pred)
    if f1>best_f1:
        best_f1=f1
        best_t=t

print("best_threshold:",best_t)
print("best_f1:",round(best_f1,4))

best_threshold: 0.59
best_f1: 0.4698


In [28]:
# Final Evaluation at chosen threshold
from sklearn.metrics import classification_report,confusion_matrix

rf_pred=(rf_prob>=best_t).astype(int)

print(classification_report(y_test,rf_pred,digits=4))
print(confusion_matrix(y_test,rf_pred))

              precision    recall  f1-score   support

           0     0.8255    0.9965    0.9030      1149
           1     0.9646    0.3105    0.4698       351

    accuracy                         0.8360      1500
   macro avg     0.8951    0.6535    0.6864      1500
weighted avg     0.8581    0.8360    0.8016      1500

[[1145    4]
 [ 242  109]]


In [29]:
# Explainability (feature importance)
import pandas as pd

imp=pd.Series(rf.feature_importances_,index=X.columns).sort_values(ascending=False)
print(imp.head(20).to_string())

previous_fraud_flag      0.156553
avg_amount_7d            0.116568
max_amount_24h           0.116343
account_age_days         0.115558
transaction_amount       0.115008
is_international         0.112965
customer_age             0.084813
txn_hour                 0.076350
merchant_fraud_rate      0.046005
txn_dayofweek            0.027786
txn_day                  0.027468
is_weekend               0.004581
channel                  0.000000
currency                 0.000000
merchant_category        0.000000
transaction_timestamp    0.000000
payment_method           0.000000
customer_region          0.000000
device_type              0.000000
txn_count_24h            0.000000


In [30]:
#SAVING IT
imp.head(50).to_csv("feature_importance_top50.csv")
print("feature_importance_top50.csv")

feature_importance_top50.csv


In [31]:
# Score ALL transactions + Risk Bands
rf_all_prob=rf.predict_proba(X)[:,1]

scored=pd.DataFrame({"fraud_probability":rf_all_prob})

scored["risk_band"]=pd.cut(
    scored["fraud_probability"],
    bins=[-0.01,0.3,0.7,1.01],
    labels=["Low","Medium","High"]
)

print(scored["risk_band"].value_counts().to_dict())

{'Low': 2373, 'Medium': 2271, 'High': 356}


In [32]:
# Reason Codes (Top 3 drivers per transaction)
# Normalizing feature values, Multiply by feature importance, pick top 3
X_norm=(X-X.min())/(X.max()-X.min()+1e-9)

weighted=X_norm.mul(imp,axis=1)

top3=weighted.apply(lambda r:list(r.sort_values(ascending=False).head(3).index),axis=1)

scored["reason_codes"]=top3.apply(lambda x:", ".join(x))

print(scored.head(5).to_string(index=False))

 fraud_probability risk_band                                        reason_codes
          0.323875    Medium    is_international, account_age_days, customer_age
          0.346284    Medium is_international, customer_age, merchant_fraud_rate
          0.239083       Low        is_international, txn_hour, account_age_days
          0.193170       Low        is_international, account_age_days, txn_hour
          0.421696    Medium is_international, customer_age, merchant_fraud_rate


In [33]:
# Creating Final Scored Output Table
final=pd.concat([df.reset_index(drop=True),scored],axis=1)

final.to_csv("scored_transactions_v1.csv",index=False)
print("scored_transactions_v1.csv")

scored_transactions_v1.csv


In [34]:
# Saving Trained Model Artifact
import joblib

joblib.dump(rf,"rf_fraud_model_v1.joblib")
joblib.dump({"threshold":best_t,"features":list(X.columns)},"model_metadata_v1.joblib")

print("rf_fraud_model_v1.joblib")
print("model_metadata_v1.joblib")

rf_fraud_model_v1.joblib
model_metadata_v1.joblib


In [37]:
#Exporting Scored Transactions to SQL Server
import pyodbc

server=r"localhost\SQLEXPRESS"
database="FraudAnalyticsDB"
conn=pyodbc.connect(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;"
)

cursor=conn.cursor()

cursor.execute("""
IF OBJECT_ID('marts.Scored_Transactions','U') IS NOT NULL DROP TABLE marts.Scored_Transactions;
""")
conn.commit()

create_cols=", ".join([f"[{c}] VARCHAR(255) NULL" for c in final.columns])
cursor.execute(f"CREATE TABLE marts.Scored_Transactions ({create_cols});")
conn.commit()

rows=[tuple(map(lambda v: None if pd.isna(v) else str(v), r)) for r in final.itertuples(index=False,name=None)]
placeholders=",".join(["?"]*len(final.columns))

cursor.fast_executemany=True
cursor.executemany(f"INSERT INTO marts.Scored_Transactions VALUES ({placeholders})",rows)
conn.commit()

print("exported_rows:",len(rows))
conn.close()



exported_rows: 5000
